# Phase 2 — Ablation Study: Original VAE + LPIPS + Bilateral Post-processing

**Workflow:**
1. Clone repo + copy data from Drive
2. Install dependencies
3. **[5 min]** Tune `lambda_p` on 5 frames (informatif seulement — on force 0.001)
4. **[~15 min]** Generate qualitative report figures (4 frames)
5. **[~3 h]** Full ablation sweep — Option 1 + Option 2 on 30 frames
6. Generate comparison tables and Pareto figure
7. Save all results to Drive

**Key change vs previous run:** lambda_p fixé à **0.001** (le run précédent utilisait 0.01
calibré sur 5 frames — trop fort, DFR effondré à 0.07 au lieu de ~0.20).

**Nouvelles métriques :** DFR proportionnel + DFR strict + ASR + mAP drop + LPIPS

**Estimated total:** ~3.5 h, ~4 Colab Pro units

## Cell 1 — Clone repo

In [ ]:
import os

REPO_DIR = '/content/pfe-latent-attack'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes ...')
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

for script in ['scripts/tune_lambda_p.py',
               'scripts/generate_qualitative.py',
               'scripts/run_ablation_sweep.py',
               'scripts/ablation_table.py',
               'configs/phase2_option1.yaml']:
    status = '✓' if os.path.exists(script) else '✗ MISSING — run git pull'
    print(f'  {status}  {script}')

## Cell 2 — Mount Drive and copy data

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
ASSETS = '/content/drive/MyDrive/pfe_assets'

os.makedirs('runs/yolov8n_detrac', exist_ok=True)
if not os.path.exists('runs/yolov8n_detrac/best.pt'):
    shutil.copy(f'{ASSETS}/best.pt', 'runs/yolov8n_detrac/best.pt')
print(f'✓ best.pt ({os.path.getsize("runs/yolov8n_detrac/best.pt")//1024} KB)')

if not os.path.exists('data/images_50'):
    shutil.copytree(f'{ASSETS}/images_50', 'data/images_50')
n50 = len([f for f in os.listdir('data/images_50') if f.endswith('.jpg')])
print(f'✓ data/images_50: {n50} frames')

for d in ['results/ablation', 'results/ablation_lp001',
          'results/qualitative/frames', 'results/qualitative/panels',
          'results/figures/png', 'results/figures/pdf', 'results/iso_budget']:
    os.makedirs(d, exist_ok=True)
print('✓ Output directories ready')

## Cell 3 — Install dependencies

In [ ]:
!pip install lpips>=0.1.4 torchmetrics -q
!pip install -r requirements.txt -q

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
      if torch.cuda.is_available() else 'N/A')

# Verify torchmetrics for mAP drop computation
try:
    from torchmetrics.detection import MeanAveragePrecision
    print('✓ torchmetrics.detection available')
except ImportError:
    print('✗ torchmetrics manquant — mAP drop ne sera pas calculé')

## Cell 4 — Tune lambda_p (~5 min) — informatif

Runs the attack on **5 frames** across 6 values of `lambda_p`.
**Note :** Le run précédent a donné 0.01 sur 5 frames, mais sur 30 frames ça
sur-contraignait l'attaque (DFR 0.20 → 0.07). On force **0.001** dans Cell 4b.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/tune_lambda_p.py \
    --config configs/phase2_option1.yaml \
    --data   data/images_50 \
    --n_frames 5 \
    --eps_z  0.50

### Cell 4b — Forcer lambda_p = 0.001

**IMPORTANT :** On ignore la recommandation du tuning sur 5 frames et on force
`lambda_p=0.001`. Raison : le tuning sur 5 frames a haute variance et a recommandé
0.01, mais sur 30 frames ce lambda sur-contraignait la perceptual loss et
effondrait le DFR de 0.20 à 0.07.

In [ ]:
import re

# Force 0.001 — NE PAS changer cette valeur sans justification expérimentale
LAMBDA_P = 0.001

config_path = 'configs/phase2_option1.yaml'
text = open(config_path).read()
text = re.sub(r'(lambda_p:\s*)[\d.e+-]+', f'\\g<1>{LAMBDA_P}', text)
open(config_path, 'w').write(text)
print(f'✓ lambda_p forcé à {LAMBDA_P} dans {config_path}')

for line in open(config_path):
    if 'lambda_p' in line:
        print(' ', line.rstrip())

## Cell 5 — Qualitative figures for the report (~15 min)

Runs the attack on **4 representative frames** and generates:
- `f15_comparison_{stem}.png` — Clean | Option 1 | Option 2 | Heatmap (4-col panel)
- `f16_filter_effect_{stem}.png` — Bilateral filter before/after
- `f17_ablation_grid.png` — Multi-frame ablation grid
- `f18_perturbation_heatmap.png` — Shared-vmax heatmap comparison

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/generate_qualitative.py \
    --config  configs/phase2_option1.yaml \
    --data    data/images_50 \
    --frames  img00001 img00008 img00020 img00026 \
    --output  results/qualitative \
    --eps_z   0.50

In [ ]:
from IPython.display import Image, display
from pathlib import Path

panels = sorted(Path('results/qualitative/panels').glob('*.png'))
print(f'Generated {len(panels)} panel figures:')
for p in panels:
    print(f'  {p.name}')
    display(Image(str(p), width=900))

## Cell 6 — Full ablation sweep (~3 h) — lambda_p=0.001

Métriques calculées par frame : **DFR proportionnel, DFR strict, ASR, mAP drop, LPIPS**

Sweeps `eps_z ∈ {0.25, 0.50, 1.00}` + PGD baselines sur **30 frames**.
Output → `results/ablation_lp001/` (séparé du run précédent à lambda_p=0.01).

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_ablation_sweep.py \
    --config  configs/phase2_option1.yaml \
    --data    data/images_50 \
    --n_frames 30 \
    --output  results/ablation_lp001 \
    --resume

In [ ]:
import json

summary = json.load(open('results/ablation_lp001/summary.json'))

print(f"{'Config':<32} {'DFR':>8} {'DFR_str':>8} {'ASR':>6} {'mAP_drop':>9} {'LPIPS':>7}")
print('-' * 75)
for tag, s in summary.items():
    dfr   = f"{s['mean_dfr']:+.4f}"        if s.get('mean_dfr')        is not None else '   N/A'
    dstr  = f"{s['dfr_strict_rate']:.3f}"  if s.get('dfr_strict_rate') is not None else '    N/A'
    asr   = f"{s['mean_asr']:.3f}"         if s.get('mean_asr')        is not None else '   N/A'
    mapd  = f"{s['mean_map_drop']:.4f}"    if s.get('mean_map_drop')   is not None else '     N/A'
    lpips = f"{s['mean_lpips']:.4f}"       if s.get('mean_lpips')      is not None else '    N/A'
    print(f"{tag:<32} {dfr:>8} {dstr:>8} {asr:>6} {mapd:>9} {lpips:>7}")

print('\nLégende: DFR_str = fraction frames n_adv==0 | ASR = toutes classes supprimées')

## Cell 7 — Generate comparison tables and figures (~2 min)

In [ ]:
!python scripts/ablation_table.py \
    --phase1_dir   results/iso_budget \
    --ablation_dir results/ablation_lp001 \
    --output_dir   results/figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for fig in ['ablation_dfr_bar.png', 'ablation_lpips_box.png', 'ablation_pareto.png']:
    p = Path('results/figures') / fig
    if p.exists():
        print(f'--- {fig} ---')
        display(Image(str(p), width=800))

## Cell 8 — Generate Pareto curve f14 (~1 min)

In [ ]:
!python scripts/generate_pareto.py \
    --summary results/ablation_lp001/summary.json

from IPython.display import Image
Image('results/figures/png/f14_pareto_dfr_lpips.png', width=700)

## Cell 9 — Print markdown table for the report

In [ ]:
import json, os

# Table markdown
table_path = 'results/figures/ablation_table.md'
if os.path.exists(table_path):
    print(open(table_path).read())
else:
    print('Table not found — run Cell 7 first.')

# Table complète avec toutes les métriques
print('\n=== Table complète toutes métriques ===')
summary = json.load(open('results/ablation_lp001/summary.json'))
print(f"{'Config':<32} {'DFR':>8} {'DFR_strict':>10} {'ASR':>6} {'mAP_drop':>9} {'LPIPS':>8} {'n_frames':>8}")
print('-' * 85)
for tag, s in summary.items():
    row = [
        f"{tag:<32}",
        f"{s.get('mean_dfr', 0):>+8.4f}",
        f"{s.get('dfr_strict_rate', 0):>10.3f}",
        f"{s.get('mean_asr', 0):>6.3f}",
        f"{s.get('mean_map_drop', 0):>9.4f}",
        f"{s.get('mean_lpips', 0):>8.4f}",
        f"{s.get('n_frames', 0):>8}",
    ]
    print(' '.join(row))

## Cell 10 — Save all results to Drive

In [ ]:
import shutil, os
from pathlib import Path

ASSETS = '/content/drive/MyDrive/pfe_assets'

# Nouveau sweep lambda_p=0.001
shutil.copytree('results/ablation_lp001',
                f'{ASSETS}/ablation_lp001',
                dirs_exist_ok=True)
print('✓ results/ablation_lp001 → Drive')

# Qualitative figures
shutil.copytree('results/qualitative',
                f'{ASSETS}/qualitative',
                dirs_exist_ok=True)
print('✓ results/qualitative → Drive')

# Statistical figures + tables
shutil.copytree('results/figures',
                f'{ASSETS}/figures',
                dirs_exist_ok=True)
print('✓ results/figures → Drive')
print('\nDone. Récupérer les résultats depuis pfe_assets/ablation_lp001/summary.json')